# Final64 Model Comparison

Notebook ini digunakan untuk membandingkan:

- Fixed Graph WaveNet
- Adaptive Graph WaveNet
- Hybrid Graph WaveNet
- GC-LSTM

Aspek yang dibandingkan:
- MAE
- RMSE
- parameter count
- inference time
- training time
- best epoch

Perbandingan dilakukan pada dataset Final64 dengan setup evaluasi yang sama.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path):
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (path / "data" / "final_64").exists() and (path / "outputs" / "final_64").exists():
            return path

    raise FileNotFoundError("Project root tidak ditemukan.")


ROOT = find_project_root(CURRENT_DIR)

print("Project root:", ROOT)

Project root: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent


In [2]:
GWN_DIR = ROOT / "outputs" / "final_64" / "graphwavenet_audited"

GCLSTM_DIR = ROOT / "outputs" / "final_64" / "gclstm_baseline"

GWN_RESULTS_PATH = GWN_DIR / "graphwavenet_multiseed_all_results.csv"

GCLSTM_RESULTS_PATH = GCLSTM_DIR / "gclstm_seed42_results.csv"

GCLSTM_TIMING_PATH = GCLSTM_DIR / "gclstm_seed42_timing.csv"

print("GWN results exists :", GWN_RESULTS_PATH.exists())
print("GC-LSTM results    :", GCLSTM_RESULTS_PATH.exists())
print("GC-LSTM timing     :", GCLSTM_TIMING_PATH.exists())

GWN results exists : True
GC-LSTM results    : True
GC-LSTM timing     : True


In [3]:
gwn_results = pd.read_csv(GWN_RESULTS_PATH)

gclstm_results = pd.read_csv(GCLSTM_RESULTS_PATH)

gclstm_timing = pd.read_csv(GCLSTM_TIMING_PATH)

print("=== GWN COLUMNS ===")
print(gwn_results.columns.tolist())
print(gwn_results.head())

print("\n=== GC-LSTM RESULT COLUMNS ===")
print(gclstm_results.columns.tolist())
print(gclstm_results.head())

print("\n=== GC-LSTM TIMING COLUMNS ===")
print(gclstm_timing.columns.tolist())
print(gclstm_timing.head())

=== GWN COLUMNS ===
['model', 'horizon', 'MAE', 'RMSE', 'N', 'seed', 'mode', 'best_epoch', 'best_val_loss', 'params']
                            model horizon       MAE      RMSE     N  seed  \
0     Graph WaveNet Fixed Audited     t+1  2.497249  3.969238  2944    42   
1     Graph WaveNet Fixed Audited     t+3  2.580379  4.061025  2944    42   
2     Graph WaveNet Fixed Audited     t+6  2.573513  4.135124  3008    42   
3  Graph WaveNet Adaptive Audited     t+1  2.282072  3.836623  2944    42   
4  Graph WaveNet Adaptive Audited     t+3  2.501486  4.026375  2944    42   

       mode  best_epoch  best_val_loss  params  
0     fixed         NaN            NaN     NaN  
1     fixed         NaN            NaN     NaN  
2     fixed         NaN            NaN     NaN  
3  adaptive         NaN            NaN     NaN  
4  adaptive         NaN            NaN     NaN  

=== GC-LSTM RESULT COLUMNS ===
['model', 'seed', 'horizon', 'mae', 'rmse', 'valid_targets']
     model  seed horizon       m

## Prepare Seed-42 Comparison

Perbandingan awal dilakukan pada seed 42 agar seluruh model dibandingkan pada kondisi seed yang sama.

Model yang dibandingkan:
- Fixed Graph WaveNet
- Adaptive Graph WaveNet
- Hybrid Graph WaveNet
- GC-LSTM

Metrik awal:
- MAE
- RMSE
- parameter count
- best epoch

Informasi parameter dan best epoch Graph WaveNet dilengkapi dari hasil training seed 42 sebelumnya karena kolom tersebut belum tersimpan pada file hasil multi-seed.

In [4]:
# ============================================================
# PREPARE SEED-42 RESULTS
# ============================================================

# Ambil hanya seed 42 dari Graph WaveNet
gwn_seed42 = gwn_results[gwn_results["seed"] == 42].copy()

# Rapikan nama model
model_name_map = {"fixed": "Fixed GWN", "adaptive": "Adaptive GWN", "hybrid": "Hybrid GWN"}

gwn_seed42["model_clean"] = gwn_seed42["mode"].map(model_name_map)

# Metadata dari hasil training seed 42 sebelumnya
gwn_params = {"fixed": 276291, "adaptive": 261187, "hybrid": 293955}

gwn_best_epoch = {"fixed": 58, "adaptive": 78, "hybrid": 85}

gwn_seed42["parameters"] = gwn_seed42["mode"].map(gwn_params)

gwn_seed42["best_epoch_clean"] = gwn_seed42["mode"].map(gwn_best_epoch)

# Format Graph WaveNet
gwn_clean = pd.DataFrame(
    {
        "model": gwn_seed42["model_clean"],
        "seed": gwn_seed42["seed"],
        "horizon": gwn_seed42["horizon"],
        "mae": gwn_seed42["MAE"],
        "rmse": gwn_seed42["RMSE"],
        "valid_targets": gwn_seed42["N"],
        "parameters": gwn_seed42["parameters"],
        "best_epoch": gwn_seed42["best_epoch_clean"],
    }
)

# Format GC-LSTM
gc_params = int(gclstm_timing.loc[0, "parameters"])

gc_best_epoch = int(gclstm_timing.loc[0, "best_epoch"])

gclstm_clean = gclstm_results.copy()

gclstm_clean["parameters"] = gc_params
gclstm_clean["best_epoch"] = gc_best_epoch

gclstm_clean = gclstm_clean[
    ["model", "seed", "horizon", "mae", "rmse", "valid_targets", "parameters", "best_epoch"]
]

# Gabungkan
seed42_comparison = pd.concat([gwn_clean, gclstm_clean], ignore_index=True)

print("=== SEED 42 COMPARISON ===")
display(seed42_comparison)

=== SEED 42 COMPARISON ===


,model,seed,horizon,mae,rmse,valid_targets,parameters,best_epoch
0,Fixed GWN,42,t+1,2.497249,3.969238,2944,276291,58
1,Fixed GWN,42,t+3,2.580379,4.061025,2944,276291,58
2,Fixed GWN,42,t+6,2.573513,4.135124,3008,276291,58
3,Adaptive GWN,42,t+1,2.282072,3.836623,2944,261187,78
4,Adaptive GWN,42,t+3,2.501486,4.026375,2944,261187,78
5,Adaptive GWN,42,t+6,2.538214,4.022828,3008,261187,78
6,Hybrid GWN,42,t+1,2.223365,3.762612,2944,293955,85
7,Hybrid GWN,42,t+3,2.334818,3.879791,2944,293955,85
8,Hybrid GWN,42,t+6,2.357840,3.785689,3008,293955,85
9,GC-LSTM,42,t+1,2.023939,3.872310,2944,553472,86


## Ringkasan Performa Seed 42

Hasil per horizon diringkas menjadi rata-rata MAE dan RMSE untuk setiap model.

Ringkasan ini digunakan untuk melihat trade-off antara:
- akurasi prediksi,
- jumlah parameter,
- best epoch,
- dan selanjutnya inference time.

In [5]:
# ============================================================
# SUMMARY PER MODEL — SEED 42
# ============================================================

seed42_summary = seed42_comparison.groupby("model", as_index=False).agg(
    avg_mae=("mae", "mean"),
    avg_rmse=("rmse", "mean"),
    parameters=("parameters", "first"),
    best_epoch=("best_epoch", "first"),
)

# urutan model
model_order = ["Fixed GWN", "Adaptive GWN", "Hybrid GWN", "GC-LSTM"]

seed42_summary["model"] = pd.Categorical(
    seed42_summary["model"], categories=model_order, ordered=True
)

seed42_summary = seed42_summary.sort_values("model").reset_index(drop=True)

print("=== SEED 42 MODEL SUMMARY ===")
display(seed42_summary)

=== SEED 42 MODEL SUMMARY ===


,model,avg_mae,avg_rmse,parameters,best_epoch
0,Fixed GWN,2.550380,4.055129,276291,58
1,Adaptive GWN,2.440591,3.961942,261187,78
2,Hybrid GWN,2.305341,3.809364,293955,85
3,GC-LSTM,2.139079,4.070951,553472,86


## Computational Cost: Inference Time

Inference time dibandingkan menggunakan prosedur yang sama untuk seluruh model:

- input: seluruh 93 sample test Final64;
- warm-up dilakukan sebelum pengukuran;
- inference diulang sebanyak 20 kali;
- model dipanggil langsung dengan `training=False`;
- nilai yang dicatat adalah mean dan standard deviation waktu inference.

Waktu per sample juga dihitung untuk mempermudah perbandingan efisiensi antar-model.

> Catatan: benchmark ini merupakan hasil pada perangkat lokal yang sama, sehingga digunakan sebagai perbandingan relatif antar-model pada environment yang sama.

In [6]:
# ============================================================
# FAIR INFERENCE BENCHMARK
# ============================================================

inference_benchmark = pd.DataFrame(
    [
        {
            "model": "Fixed GWN",
            "inference_sec_93_samples": 0.330161,
            "inference_std_sec": 0.076620,
            "inference_ms_per_sample": 3.550116,
        },
        {
            "model": "Adaptive GWN",
            "inference_sec_93_samples": 0.296652,
            "inference_std_sec": 0.095593,
            "inference_ms_per_sample": 3.189805,
        },
        {
            "model": "Hybrid GWN",
            "inference_sec_93_samples": 0.305737,
            "inference_std_sec": 0.058958,
            "inference_ms_per_sample": 3.287493,
        },
        {
            "model": "GC-LSTM",
            "inference_sec_93_samples": 0.033850,
            "inference_std_sec": 0.002532,
            "inference_ms_per_sample": 0.364000,
        },
    ]
)

display(inference_benchmark)

,model,inference_sec_93_samples,inference_std_sec,inference_ms_per_sample
0,Fixed GWN,0.330161,0.076620,3.550116
1,Adaptive GWN,0.296652,0.095593,3.189805
2,Hybrid GWN,0.305737,0.058958,3.287493
3,GC-LSTM,0.033850,0.002532,0.364000


## Seed-42 Accuracy and Efficiency Comparison

Tabel berikut menggabungkan performa prediksi dan kompleksitas komputasi pada seed 42.

Aspek yang dibandingkan:
- rata-rata MAE;
- rata-rata RMSE;
- jumlah parameter;
- best epoch;
- inference time untuk 93 test samples;
- inference time per sample.

In [7]:
# ============================================================
# MERGE ACCURACY + COMPLEXITY + INFERENCE
# ============================================================

final_seed42_comparison = seed42_summary.merge(inference_benchmark, on="model", how="left")

# Biar tabel lebih enak dibaca
final_seed42_comparison["avg_mae"] = final_seed42_comparison["avg_mae"].round(4)

final_seed42_comparison["avg_rmse"] = final_seed42_comparison["avg_rmse"].round(4)

final_seed42_comparison["inference_sec_93_samples"] = final_seed42_comparison[
    "inference_sec_93_samples"
].round(4)

final_seed42_comparison["inference_std_sec"] = final_seed42_comparison["inference_std_sec"].round(4)

final_seed42_comparison["inference_ms_per_sample"] = final_seed42_comparison[
    "inference_ms_per_sample"
].round(4)

print("=== SEED 42 ACCURACY + EFFICIENCY COMPARISON ===")

display(
    final_seed42_comparison[
        [
            "model",
            "avg_mae",
            "avg_rmse",
            "parameters",
            "best_epoch",
            "inference_sec_93_samples",
            "inference_std_sec",
            "inference_ms_per_sample",
        ]
    ]
)

=== SEED 42 ACCURACY + EFFICIENCY COMPARISON ===


,model,avg_mae,avg_rmse,parameters,best_epoch,inference_sec_93_samples,inference_std_sec,inference_ms_per_sample
0,Fixed GWN,2.5504,4.0551,276291,58,0.3302,0.0766,3.5501
1,Adaptive GWN,2.4406,3.9619,261187,78,0.2967,0.0956,3.1898
2,Hybrid GWN,2.3053,3.8094,293955,85,0.3057,0.0590,3.2875
3,GC-LSTM,2.1391,4.0710,553472,86,0.0338,0.0025,0.3640


## Interpretasi Awal Seed 42

Hasil seed 42 menunjukkan adanya trade-off antara akurasi, kompleksitas model, dan kecepatan inference.

- GC-LSTM menghasilkan rata-rata MAE paling rendah dan inference paling cepat, tetapi memiliki jumlah parameter paling besar.
- Hybrid Graph WaveNet menghasilkan rata-rata RMSE paling rendah dengan jumlah parameter yang masih lebih kecil dibanding GC-LSTM.
- Adaptive Graph WaveNet memiliki jumlah parameter paling sedikit dan inference paling cepat di antara konfigurasi Graph WaveNet.
- Fixed Graph WaveNet memiliki error dan inference time yang relatif lebih tinggi dibanding konfigurasi Graph WaveNet lainnya.

Hasil ini menunjukkan bahwa pemilihan model final tidak dapat hanya didasarkan pada satu metrik. Evaluasi perlu mempertimbangkan akurasi, kompleksitas, dan efisiensi komputasi secara bersamaan.

## Final Comparison: Accuracy, Complexity, Training Cost, and Inference Cost

Tabel akhir menggabungkan seluruh hasil seed 42:

- rata-rata MAE;
- rata-rata RMSE;
- jumlah parameter;
- best epoch benchmark;
- rata-rata training time per epoch;
- total training time;
- inference time per sample.

Perbandingan ini digunakan untuk melihat trade-off antara akurasi dan efisiensi komputasi.

In [8]:
# ============================================================
# LOAD TRAINING COST
# ============================================================

TRAINING_COST_PATH = ROOT / "outputs" / "final_64" / "training_cost_benchmark_seed42.csv"

training_cost = pd.read_csv(TRAINING_COST_PATH)

print("Training cost loaded:")
display(training_cost)

Training cost loaded:


,model,mean_sec_per_epoch,total_training_time_sec,best_epoch_benchmark
0,Fixed GWN,3.2293,284.6711,63
1,Adaptive GWN,2.9060,267.8679,67
2,Hybrid GWN,3.8203,401.6228,80
3,GC-LSTM,0.2540,23.0799,65


In [9]:
# ============================================================
# MERGE FINAL COMPARISON
# ============================================================

final_comparison_full = final_seed42_comparison.merge(training_cost, on="model", how="left")

final_comparison_full = final_comparison_full[
    [
        "model",
        "avg_mae",
        "avg_rmse",
        "parameters",
        "best_epoch_benchmark",
        "mean_sec_per_epoch",
        "total_training_time_sec",
        "inference_ms_per_sample",
    ]
].copy()

final_comparison_full["mean_sec_per_epoch"] = final_comparison_full["mean_sec_per_epoch"].round(4)

final_comparison_full["total_training_time_sec"] = final_comparison_full[
    "total_training_time_sec"
].round(2)

final_comparison_full["inference_ms_per_sample"] = final_comparison_full[
    "inference_ms_per_sample"
].round(4)

print("=== FINAL SEED 42 COMPARISON ===")
display(final_comparison_full)

=== FINAL SEED 42 COMPARISON ===


,model,avg_mae,avg_rmse,parameters,best_epoch_benchmark,mean_sec_per_epoch,total_training_time_sec,inference_ms_per_sample
0,Fixed GWN,2.5504,4.0551,276291,63,3.2293,284.67,3.5501
1,Adaptive GWN,2.4406,3.9619,261187,67,2.9060,267.87,3.1898
2,Hybrid GWN,2.3053,3.8094,293955,80,3.8203,401.62,3.2875
3,GC-LSTM,2.1391,4.0710,553472,65,0.2540,23.08,0.3640


## GC-LSTM historis dan batas perbandingan
Label file `fair` dipertahankan untuk keterlacakan hasil lama. Audit menemukan bahwa normalisasi loss baseline historis belum identik dengan GWN. Tabel ini bersifat deskriptif; hasil loss terkoreksi hanya boleh dipakai setelah training ulang selesai.


In [10]:
# ============================================================
# LOAD FAIR GC-LSTM RESULTS
# ============================================================

GCLSTM_FAIR_RESULTS_PATH = (
    ROOT / "outputs" / "final_64" / "gclstm_baseline" / "gclstm_fair_seed42_results.csv"
)

GCLSTM_FAIR_TIMING_PATH = (
    ROOT / "outputs" / "final_64" / "gclstm_baseline" / "gclstm_fair_seed42_timing.csv"
)

gclstm_fair_results = pd.read_csv(GCLSTM_FAIR_RESULTS_PATH)

gclstm_fair_timing = pd.read_csv(GCLSTM_FAIR_TIMING_PATH)

print("=== GC-LSTM FAIR RESULTS ===")
display(gclstm_fair_results)

print("\n=== GC-LSTM FAIR TIMING ===")
display(gclstm_fair_timing)

=== GC-LSTM FAIR RESULTS ===


,model,seed,horizon,mae,rmse,valid_targets
0,GC-LSTM Fair,42,t+1,2.233555,3.623349,2944
1,GC-LSTM Fair,42,t+3,2.232332,3.724498,2944
2,GC-LSTM Fair,42,t+6,2.450448,4.109270,3008



=== GC-LSTM FAIR TIMING ===


,model,seed,parameters,epochs_run,best_epoch,mean_sec_per_epoch,total_training_time_sec,mean_inference_time_sec,std_inference_time_sec,inference_ms_per_sample
0,GC-LSTM Fair,42,553472,90,65,0.254043,23.079881,0.034504,0.002612,0.371008


In [11]:
# ============================================================
# UPDATE FINAL COMPARISON WITH FAIR GC-LSTM
# ============================================================

fair_avg_mae = gclstm_fair_results["mae"].mean()

fair_avg_rmse = gclstm_fair_results["rmse"].mean()

fair_timing = gclstm_fair_timing.iloc[0]

# copy tabel final
final_comparison_fair = final_comparison_full.copy()

# replace GC-LSTM row
mask_gc = final_comparison_fair["model"] == "GC-LSTM"

final_comparison_fair.loc[mask_gc, "avg_mae"] = fair_avg_mae

final_comparison_fair.loc[mask_gc, "avg_rmse"] = fair_avg_rmse

final_comparison_fair.loc[mask_gc, "parameters"] = fair_timing["parameters"]

final_comparison_fair.loc[mask_gc, "best_epoch_benchmark"] = fair_timing["best_epoch"]

final_comparison_fair.loc[mask_gc, "mean_sec_per_epoch"] = fair_timing["mean_sec_per_epoch"]

final_comparison_fair.loc[mask_gc, "total_training_time_sec"] = fair_timing[
    "total_training_time_sec"
]

final_comparison_fair.loc[mask_gc, "inference_ms_per_sample"] = fair_timing[
    "inference_ms_per_sample"
]

# rounding
final_comparison_fair["avg_mae"] = final_comparison_fair["avg_mae"].round(4)

final_comparison_fair["avg_rmse"] = final_comparison_fair["avg_rmse"].round(4)

final_comparison_fair["mean_sec_per_epoch"] = final_comparison_fair["mean_sec_per_epoch"].round(4)

final_comparison_fair["total_training_time_sec"] = final_comparison_fair[
    "total_training_time_sec"
].round(2)

final_comparison_fair["inference_ms_per_sample"] = final_comparison_fair[
    "inference_ms_per_sample"
].round(4)

print("=== FINAL FAIR SEED 42 COMPARISON ===")

display(
    final_comparison_fair[
        [
            "model",
            "avg_mae",
            "avg_rmse",
            "parameters",
            "best_epoch_benchmark",
            "mean_sec_per_epoch",
            "total_training_time_sec",
            "inference_ms_per_sample",
        ]
    ]
)

=== FINAL FAIR SEED 42 COMPARISON ===


,model,avg_mae,avg_rmse,parameters,best_epoch_benchmark,mean_sec_per_epoch,total_training_time_sec,inference_ms_per_sample
0,Fixed GWN,2.5504,4.0551,276291,63,3.2293,284.67,3.5501
1,Adaptive GWN,2.4406,3.9619,261187,67,2.9060,267.87,3.1898
2,Hybrid GWN,2.3053,3.8094,293955,80,3.8203,401.62,3.2875
3,GC-LSTM,2.3054,3.8190,553472,65,0.2540,23.08,0.3710


## Save Final Comparison Results

Hasil perbandingan seed 42 disimpan menjadi dua file:

1. Per-horizon comparison untuk t+1, t+3, dan t+6.
2. Summary comparison yang menggabungkan accuracy, model complexity, training cost, dan inference cost.

In [12]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

COMPARISON_DIR = ROOT / "outputs" / "final_64" / "model_comparison"

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# PER-HORIZON FAIR COMPARISON
# ============================================================

# GWN
gwn_per_horizon = seed42_comparison[seed42_comparison["model"] != "GC-LSTM"].copy()

# GC-LSTM fair
gc_fair_per_horizon = gclstm_fair_results.copy()

gc_fair_per_horizon["model"] = "GC-LSTM"
gc_fair_per_horizon["parameters"] = int(gclstm_fair_timing.loc[0, "parameters"])
gc_fair_per_horizon["best_epoch"] = int(gclstm_fair_timing.loc[0, "best_epoch"])

gc_fair_per_horizon = gc_fair_per_horizon[
    [
        "model",
        "seed",
        "horizon",
        "mae",
        "rmse",
        "valid_targets",
        "parameters",
        "best_epoch",
    ]
]

fair_per_horizon = pd.concat([gwn_per_horizon, gc_fair_per_horizon], ignore_index=True)

# ============================================================
# SAVE
# ============================================================

PER_HORIZON_PATH = COMPARISON_DIR / "seed42_fair_per_horizon.csv"

SUMMARY_PATH = COMPARISON_DIR / "seed42_fair_summary.csv"

TRAINING_COST_COPY_PATH = COMPARISON_DIR / "training_cost_seed42.csv"

fair_per_horizon.to_csv(PER_HORIZON_PATH, index=False)

final_comparison_fair.to_csv(SUMMARY_PATH, index=False)

training_cost.to_csv(TRAINING_COST_COPY_PATH, index=False)

print("Saved:")
print(PER_HORIZON_PATH)
print(SUMMARY_PATH)
print(TRAINING_COST_COPY_PATH)

Saved:
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/model_comparison/seed42_fair_per_horizon.csv
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/model_comparison/seed42_fair_summary.csv
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/model_comparison/training_cost_seed42.csv


## Final Multi-Seed Model Comparison

Perbandingan akhir dilakukan menggunakan lima random seed yang sama:

- 42
- 123
- 456
- 789
- 2026

Model yang dibandingkan:
- Fixed Graph WaveNet
- Adaptive Graph WaveNet
- Hybrid Graph WaveNet
- GC-LSTM Fair

Evaluasi menggunakan mean dan standard deviation MAE/RMSE antar-seed pada horizon t+1, t+3, dan t+6.

In [ ]:
# ============================================================
# FINAL 5-SEED COMPARISON
# ============================================================

GWN_PATH = (
    ROOT
    / "outputs"
    / "final_64"
    / "graphwavenet_audited"
    / "graphwavenet_multiseed_all_results.csv"
)

GCLSTM_PATH = (
    ROOT / "outputs" / "final_64" / "gclstm_baseline" / "gclstm_fair_multiseed_results.csv"
)

gwn = pd.read_csv(GWN_PATH)
gc = pd.read_csv(GCLSTM_PATH)

# standardize column names
gwn = gwn.rename(columns={"MAE": "mae", "RMSE": "rmse"})


# friendly model names
def rename_gwn_model(row):
    mode = str(row["mode"]).lower()

    if mode == "fixed":
        return "Fixed GWN"

    if mode == "adaptive":
        return "Adaptive GWN"

    if mode == "hybrid":
        return "Hybrid GWN"

    return row["model"]


gwn["model_clean"] = gwn.apply(rename_gwn_model, axis=1)

gc["model_clean"] = "GC-LSTM Fair"

# combine
combined = pd.concat(
    [
        gwn[["model_clean", "seed", "horizon", "mae", "rmse"]],
        gc[["model_clean", "seed", "horizon", "mae", "rmse"]],
    ],
    ignore_index=True,
)

# ============================================================
# PER-HORIZON SUMMARY
# ============================================================

final_multiseed_summary = (
    combined.groupby(["model_clean", "horizon"])
    .agg(
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
    )
    .reset_index()
)

print("=== FINAL 5-SEED PER-HORIZON COMPARISON ===")

display(final_multiseed_summary.round(4))

# ============================================================
# OVERALL SUMMARY
# Average horizons within each seed, then calculate SD across seeds.
# ============================================================

overall_summary = (
    combined.groupby(["model_clean", "seed"], as_index=False)
    .agg(mae=("mae", "mean"), rmse=("rmse", "mean"))
    .groupby("model_clean")
    .agg(
        avg_mae=("mae", "mean"),
        avg_mae_std=("mae", "std"),
        avg_rmse=("rmse", "mean"),
        avg_rmse_std=("rmse", "std"),
    )
    .reset_index()
)

overall_summary = overall_summary.sort_values("avg_mae")

print("\n=== FINAL 5-SEED OVERALL SUMMARY ===")

display(overall_summary.round(4))

In [14]:
FINAL_DIR = ROOT / "outputs" / "final_64" / "model_comparison"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_multiseed_summary.to_csv(FINAL_DIR / "final_5seed_per_horizon_comparison.csv", index=False)

overall_summary.to_csv(FINAL_DIR / "final_5seed_overall_comparison.csv", index=False)

print("Saved.")

Saved.
